# Test de différentes méthodes de complétion des données

## Imports & Variables

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import seaborn as sns
os.environ["KERAS_BACKEND"] = "torch"
import sys
sys.path.append(os.path.abspath("../../"))
from keras.models import load_model

from src.methodes import *
from src.visualisations import *
from src.data import *
from src.evaluations import *

valeur_de_travail = 'T_Q'

fichier_nappe = "../../data/fusion/data_03288X0042_P.csv"
dossier_nappe = "../../data/fusion"

df = charger_fichier(fichier_nappe)

In [ ]:
def classifier_nappe_fluctuation(df, col="niveau_nappe_eau"):

    niveau = df[col].dropna()

    # amplitude totale
    amplitude = niveau.max() - niveau.min()

    # variations mensuelles
    variations = niveau.diff().dropna()

    variation_moy = variations.abs().mean()
    variabilite = variations.std()

    # pente moyenne (tendance globale)
    x = np.arange(len(niveau))
    slope = np.polyfit(x, niveau, 1)[0]

    # indice de dynamique
    indice = variation_moy + variabilite

    # classification
    if indice > 1.0:
        type_nappe = "nappe réactive"
    elif indice > 0.4:
        type_nappe = "nappe intermédiaire"
    else:
        type_nappe = "nappe inertielle"

    return {
        "amplitude": amplitude,
        "variation_moyenne": variation_moy,
        "variabilite": variabilite,
        "pente": slope,
        "indice_dynamique": indice,
        "type_nappe": type_nappe
    }

In [ ]:
resultat = classifier_nappe_fluctuation(df)

print(resultat)

In [ ]:
def plot_nappes(dossier, valeur_de_travail="niveau_nappe_eau"):
    fichiers = glob.glob(os.path.join(dossier, "*.csv"))

    for fichier in fichiers:
        try:
            df = pd.read_csv(fichier, sep=";")

            # Conversion du temps
            df['time'] = pd.to_datetime(df['time'])
            df = df.sort_values('time')

            # Conversion de la valeur
            df[valeur_de_travail] = pd.to_numeric(df[valeur_de_travail], errors='coerce')

            # Plot
            plt.figure(figsize=(10,5))
            plt.plot(df['time'], df["PRELIQ_Q"])
            plt.plot(df['time'], df[valeur_de_travail])
            plt.title(f"Nappe : {os.path.basename(fichier)}")
            plt.xlabel("Temps")
            plt.ylabel(valeur_de_travail)
            plt.grid(True)

            plt.show()

        except Exception as e:
            print(f"Erreur avec {fichier} : {e}")

In [ ]:
# plot_nappes(dossier_nappe)

## Resultats

### Interpolations

In [ ]:
arr_linear = interpolation_lineaire_array(df, valeur_de_travail)
arr_cubic = interpolation_cubique_array(df, valeur_de_travail)
arr_poly = interpolation_polynomiale_array(df, valeur_de_travail)
arr_spline = interpolation_spline_array(df, valeur_de_travail)
arr_pchip = interpolation_pchip_array(df, valeur_de_travail)
arr_akima = interpolation_akima_array(df, valeur_de_travail)

methods = {
    "Linear": arr_linear,
    "PCHIP": arr_pchip,
    "Akima": arr_akima,
    "Poly": arr_poly,
    "Cubic": arr_cubic,
    "Spline B": arr_spline,
    "Mesures": df[valeur_de_travail],
}

plt.figure(figsize=(12,6))
for label, arr in methods.items():
    plt.plot(df['time'], arr, label=label)
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()

### Réseaux de neuronnes

In [ ]:
# 1. Chargement et Scaler (Sur les 19k lignes)
features = ["niveau_nappe_eau","lon","lat","time_num","ETP_Q","PRELIQ_Q","T_Q","surface_imp","surface_totale"]

# 2. Chargement des modèles avec la perte masquée
print("Chargement des modèles...")

df['time_num'] = df['time'].astype('int64') // 10**9
features_scaler = [f if f != 'time' else 'time_num' for f in features]
mon_scaler = joblib.load('../../scaler/scaler.save')

custom_objs = {'masked_mse': masked_mse}
model_lstm = load_model("../../models/LSTM.keras", custom_objects={'masked_mse': masked_mse})
model_bilstm = load_model("../../models/BILSTM.keras", custom_objects={'masked_mse': masked_mse})

arr_lstm = lstm_predict_array(df, model_lstm, mon_scaler, features, window_size=24, target_col=valeur_de_travail)
arr_bilstm = lstm_predict_array(df, model_bilstm, mon_scaler, features, window_size=24, target_col=valeur_de_travail)

methods = {
    "LSTM": arr_lstm,
    "BILSTM": arr_bilstm,
    "Mesures": df[valeur_de_travail],
}

# 5. Graphique
plt.figure(figsize=(12,6))
for label, arr in methods.items():
    plt.plot(df['time'], arr, label=label, alpha=0.7)

plt.title(f"Comparaison LSTM vs BILSTM sur {valeur_de_travail}")
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

### Autres méthodes

In [ ]:
arr_rf = random_forest_delta_array(df, valeur_de_travail)
knn = knn_impute(df, valeur_de_travail, k=7)
BTS = bootstrap_saisonnier_impute(df, valeur_de_travail)

methods = {
    "knn": knn,
    "bts" : BTS,
    "RF": arr_rf,
    "Mesures": df[valeur_de_travail],

}

plt.figure(figsize=(12,6))
for label, arr in methods.items():
    plt.plot(df['time'],arr, label=label, alpha=0.7)
plt.xlabel("Date")
plt.ylabel(valeur_de_travail)
plt.legend()
plt.show()

## Evaluation des méthodes

### MAE - Mean Absolute Error

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|
$$

Où :

- $y_i$ : valeur observée  
- $\hat{y}_i$ : valeur prédite  
- $n$ : nombre d'observations  

In [ ]:
def nmae(prediction, reelle):
    range_val = np.max(reelle)
    if range_val == 0: return 0 # Sécurité si le signal est plat
    return np.mean(np.abs(reelle - prediction)) / range_val

### RMSE - Root Mean Square Error

$$
\text{RMSE} = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2 }
$$

Où :

- $y_i$ : valeur observée  
- $\hat{y}_i$ : valeur prédite  
- $n$ : nombre d'observations  

In [ ]:
def nrmse(prediction, reelle):
    range_val = np.max(reelle)
    if range_val == 0: return 0
    # On calcule la RMSE standard puis on normalise
    return np.sqrt(np.mean((reelle - prediction) ** 2)) / range_val

### NSE - Nash–Sutcliffe Efficiency

$$
\text{NSE} = 1 - 
\frac{
\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2
}{
\sum_{i=1}^{n} \left( y_i - \bar{y} \right)^2
}
$$

Où :

- $y_i$ : valeur observée  
- $\hat{y}_i$ : valeur prédite  
- $\bar{y}$ : moyenne des valeurs observées  
- $n$ : nombre d'observations  

In [ ]:
def nse(prediction, reelle):    
    numerator = np.sum((reelle - prediction) ** 2)
    denominator = np.sum((reelle - np.mean(reelle)) ** 2)
    
    if denominator == 0:
        return np.nan
    
    return 1 - (numerator / denominator)

### Visuels

In [ ]:
def compute_ranking_stats(error_df):
    error_df = error_df.copy()
    error_df['rank'] = error_df.groupby(['dataset', 'pct_removed'])['NMAE'].rank(method='min')

    stats = []

    methods = error_df['method'].unique()

    for method in methods:
        m_df = error_df[error_df['method'] == method]
        last_rank_per_group = error_df.groupby(['dataset', 'pct_removed'])['rank'].transform('max')
        stats.append({
            'method': method,
            'Premier': (m_df['rank'] == 1).sum(),
            'Dernier': (m_df['rank'] == last_rank_per_group[m_df.index]).sum(),
            'rank_moyen': m_df['rank'].mean()
        })

    return pd.DataFrame(stats)

def run_full_evaluation(folder_path,
                        valeur_de_travail,
                        fichier_scaler,
                        remove_pct_list=[0.1],
                        random_state=42,
                        target_dataset_name=None,
                        path="../../"):
    
    print(f"""
============================================================================\n
\n
                            {valeur_de_travail}\n
\n
============================================================================\n
""")

    error_df = evaluate_all_files(
        folder_path,
        valeur_de_travail,
        joblib.load(fichier_scaler),
        remove_pct_list,
        random_state,
        path=path
    )

    if error_df is None:
        return None

    processed_count = error_df['dataset'].nunique()

    # Graph 1 & 2
    plot_scatter_zooms(error_df, processed_count)

    # Ranking
    stats_df = compute_ranking_stats(error_df)

    # Graph 3
    plot_first_vs_last(stats_df, processed_count)

    # Graph 4
    plot_first_vs_avg_rank(stats_df, processed_count, valeur_de_travail)

    plot_mean_errors_by_method(error_df, processed_count, valeur_de_travail)

    # Graph 5
    if target_dataset_name is None:
        target_dataset_name = error_df['dataset'].iloc[0]

    plot_error_heatmaps(error_df, error_metrics=['NMAE', 'NRMSE'])

    

    # plot_dataset_focus(error_df, target_dataset_name)

    return error_df, stats_df


In [ ]:
# Suppose que tu as plusieurs DataFrames
# run_full_evaluation(dossier_nappe, valeur_de_travail, remove_pct_list=[0.1,0.3,0.5,0.8])

for elt in ["niveau_nappe_eau","ETP_Q","PRELIQ_Q","T_Q"] : run_full_evaluation(dossier_nappe, elt,"../../scaler/scaler.save" , remove_pct_list=[0.3], path="../../") 


In [ ]:
def plot_dataset_predictions(df, y_full, y_missing, y_pred_dict, ds_name, method_name):
    """
    Affiche les courbes pour un dataset :
    - données réelles
    - données avec trous
    - données interpolées pour une méthode donnée
    """
    plt.figure(figsize=(12, 6))

    # Courbe prédite pour la méthode choisie
    if method_name not in y_pred_dict:
        print(f"⚠️ Méthode {method_name} non trouvée dans y_pred_dict.")
        return

    dico = {
        "PRELIQ_Q" : "le total d'eau de pluie",
        "niveau_nappe_eau" : "le niveau d'eau de la nappe",
        "T_Q" : "la température"
    }


    plt.plot(df['time'], y_full, linestyle='--', label='Données manquantes', color='black', linewidth=2, alpha= 0.4)
    plt.plot(df['time'], y_missing, label='Données réelles', color='black', linewidth=2)
    plt.plot(df['time'], y_pred_dict[method_name], label=f'Prédiction ({method_name})', color='orange', linewidth=2, alpha= 0.8)

    plt.xlabel('Time')
    plt.ylabel('Valeur')
    plt.title(f"Interpolation sur le{dico[ds_name]} - Méthode : {method_name}")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

        #"Linear": interpolation_lineaire_array(df_with_holes, valeur_de_travail),
        #"PCHIP": interpolation_pchip_array(df_with_holes, valeur_de_travail),
        #"Akima": interpolation_akima_array(df_with_holes, valeur_de_travail),
        #"Cubic": interpolation_cubique_array(df_with_holes, valeur_de_travail),
        #"Poly": interpolation_polynomiale_array(df_with_holes, valeur_de_travail),
        #"B-Spline": interpolation_spline_array(df_with_holes, valeur_de_travail),
        "KNN": knn_impute(df_with_holes, valeur_de_travail),
        "Bootstrap saisonier": bootstrap_saisonnier_impute(df_with_holes, valeur_de_travail),
        "Random forest": random_forest_delta_array(df_with_holes, valeur_de_travail),
        "LSTM" : lstm_predict_array(df_with_holes, models["LSTM"], mon_scaler, features, window_size=60, target_col=valeur_de_travail),
        "BILSTM" : lstm_predict_array(df_with_holes, models["BILSTM"], mon_scaler, features, window_size=6, target_col=valeur_de_travail),
        "LSTM2" : lstm_predict_array(df_with_holes, models["LSTM2"], mon_scaler, features, window_size=60, target_col=valeur_de_travail),
        "CNN" : lstm_predict_array(df_with_holes, models["CNN"], mon_scaler, features, window_size=6, target_col=valeur_de_travail),

In [ ]:
file = "../data/fusion/data_03276X0009_P.csv"
remove_pct = 0.0
rng = np.random.default_rng(42)

models = {
    "LSTM" : load_model("./models/LSTM.keras", custom_objects={'masked_mse': masked_mse}),
    "LSTM2" : load_model("./models/LSTM2.keras", custom_objects={'masked_mse': masked_mse}),
    "CNN" : load_model("./models/CNN.keras", custom_objects={'masked_mse': masked_mse}),
    "BILSTM" : load_model("./models/BILSTM.keras", custom_objects={'masked_mse': masked_mse})
}

valeur_de_travail = "PRELIQ_Q"

prepared = prepare_dataset(file, valeur_de_travail, remove_pct, rng, 1995, 1997)
if prepared:
    df, y_full, y_missing, ds_name = prepared
    y_pred_dict = compute_interpolations(df, y_missing, valeur_de_travail, models)

    # Affichage pour la méthode "Linear"
    plot_dataset_predictions(df, y_full, y_missing, y_pred_dict, valeur_de_travail, method_name="LSTM")